# 02 · Train the baseline  (P1)

AASIST head on **clean** cached features.  No augmentation.

This model is **not** a stepping stone to be discarded once the augmented one
works.  It is the control condition that the entire pitch rests on: the
argument is the *gap* between the two, and you cannot show a gap with one
model.  Tag it and keep it loadable forever.

In [ ]:
# --- Colab setup -----------------------------------------------------------
# Run this first in every notebook.  Idempotent.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Keep the repo and all caches on Drive so a disconnect does not cost you
    # the feature extraction pass.
    PROJECT = Path("/content/drive/MyDrive/voice-integrity")
    if not PROJECT.exists():
        raise SystemExit(
            f"Upload or clone the repo to {PROJECT} first.\n"
            "  !git clone <your-repo-url> /content/drive/MyDrive/voice-integrity"
        )
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

# Repo-local model cache.  Set BEFORE importing transformers, or it will use
# the default location and the cache will not be portable to the demo machine.
os.environ["HF_HOME"] = str(PROJECT / "cache" / "huggingface")
os.environ["TORCH_HOME"] = str(PROJECT / "cache" / "torch")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("project:", PROJECT)
print("python :", sys.version.split()[0])

In [ ]:
if IN_COLAB:
    !pip install -q transformers speechbrain soundfile librosa pydantic pyyaml cryptography wandb
    !apt-get -qq install -y ffmpeg libopencore-amrnb-dev > /dev/null

# AMR-NB encoding is the one that silently goes missing.  If this prints
# nothing, your mobile-codec augmentation does nothing and the whole
# codec-robustness result quietly evaporates.
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -i amr || echo "AMR-NB ENCODER MISSING"

In [ ]:
import torch
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device         :", torch.cuda.get_device_name(0))
    print("memory         : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from vif.common.config import load_config
from vif.data.manifests import read_manifest
from vif.models.aasist import count_parameters
from vif.models.heads import build_head

config = load_config("configs")
train_items = read_manifest("data/features/train_clean/manifest.jsonl")
dev_items   = read_manifest("data/features/dev_clean/manifest.jsonl")

head = build_head(config.model.head, feat_dim=config.model.frontend.hidden_dim)
print(f"head: {config.model.head.arch}, {count_parameters(head):,} trainable parameters")

### Why model selection is on EER, not loss

The corpus carries roughly nine spoofed utterances per bonafide one, so both
loss and accuracy track the majority class.  Selecting on either would happily
pick a model that never predicts "bonafide".

In [ ]:
from vif.train.loop import TrainConfig, train_head

train_config = TrainConfig(
    epochs=25,
    batch_size=32,
    learning_rate=1e-4,
    class_weighting=True,      # counteracts the 9:1 imbalance
    early_stop_patience=6,
)

history = train_head(
    head,
    train_items, dev_items,
    train_features="data/features/train_clean",
    dev_features="data/features/dev_clean",
    train_config=train_config,
    device=DEVICE,
    checkpoint_path="models/checkpoints/baseline.pt",
    checkpoint_meta={
        "arch": config.model.head.arch,
        "feat_dim": config.model.frontend.hidden_dim,
        "frontend_id": config.model.frontend.model_id,
        "window_samples": config.model.audio.window_samples,
        "condition": "baseline",
    },
)
print(f"\nbest dev EER {history.best_eer*100:.2f}% at epoch {history.best_epoch+1}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(history.train_loss, label="train loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.legend(); ax1.grid(alpha=.3)
ax2.plot([e*100 for e in history.dev_eer], label="dev EER %")
ax2.plot([t*100 for t in history.dev_tpr], label="dev TPR@1%FPR")
ax2.axvline(history.best_epoch, ls="--", c="k", lw=1, label="selected")
ax2.set_xlabel("epoch"); ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Calibrate on dev — and only dev

After Platt scaling the log-likelihood ratio is just `a * score + b`.  That
affine form is exactly what makes evidence **additive** at fusion time.

Fitting on the evaluation split produces numbers that look excellent and mean
nothing, so the loader refuses any calibration marked `fitted_on: eval`.

In [ ]:
from torch.utils.data import DataLoader
from vif.data.datasets import FeatureDataset, collate_features
from vif.eval.calibration import Calibrator, fit_platt
from vif.train.loop import predict

dev_loader = DataLoader(
    FeatureDataset("data/features/dev_clean", dev_items, max_frames=208),
    batch_size=32, shuffle=False, collate_fn=collate_features,
)
labels, scores = predict(head, dev_loader, DEVICE)

params = fit_platt(labels, scores, branch="spoof", split="dev")
Calibrator({"spoof": params}).save("configs/calibration.json")
print(f"llr = {params.a:.4f} * score + {params.b:.4f}")

## Exit criteria for P1

- A strong in-domain EER **and** a visibly worse out-of-domain EER, both
  recorded.
- `baseline.pt` tagged and kept.

The gap is expected.  Report it rather than tuning it away — see notebook 04.